# AquaScope Quickstart Tutorial

This notebook walks through a complete AquaScope workflow:

1. **Collect** water data from multiple sources
2. **Assess** data quality and preprocess
3. **Explore** with automated EDA
4. **Recommend** research methodologies with AI
5. **Execute** a methodology pipeline

---

## Setup

```bash
pip install -e ".[all]"
```

In [ ]:
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

## 1. Data Collection

AquaScope provides collectors for 7 data sources. Each collector follows the same pattern:

```python
collector = SomeCollector(api_key="...")
records = collector.collect()  # fetch + normalise
```

For this tutorial, we'll generate synthetic data that mimics Taiwan river monitoring data.

In [ ]:
# Generate synthetic water quality data (simulating Taiwan MOENV river data)
np.random.seed(42)

stations = {
    "ST001": "Tamsui River - Guandu Bridge",
    "ST002": "Tamsui River - Zhongxiao Bridge",
    "ST003": "Xindian River - Xiulang Bridge",
    "ST004": "Keelung River - Jiangbei Bridge",
    "ST005": "Dahan River - Sanying Bridge",
}

params = {
    "DO": (5.0, 2.0),      # Dissolved Oxygen (mg/L)
    "BOD5": (3.0, 1.5),    # Biochemical Oxygen Demand
    "COD": (8.0, 3.0),     # Chemical Oxygen Demand
    "NH3-N": (1.0, 0.8),   # Ammonia Nitrogen
    "SS": (25.0, 15.0),    # Suspended Solids
    "pH": (7.2, 0.3),      # pH
}

rows = []
base_date = datetime(2020, 1, 15)
for month in range(60):  # 5 years of monthly data
    dt = base_date + timedelta(days=30 * month)
    for sid, sname in stations.items():
        for param, (mean, std) in params.items():
            # Add a slight upward trend for BOD5 and downward for DO
            trend = month * 0.01 if param == "BOD5" else (-month * 0.005 if param == "DO" else 0)
            value = max(0, np.random.normal(mean + trend, std))
            rows.append({
                "source": "taiwan_moenv",
                "station_id": sid,
                "station_name": sname,
                "parameter": param,
                "value": round(value, 2),
                "unit": "mg/L" if param != "pH" else "pH",
                "sample_datetime": dt.isoformat(),
                "basin": "Tamsui River",
                "county": "New Taipei City",
            })

df = pd.DataFrame(rows)
print(f"Dataset: {len(df):,} records, {df['station_id'].nunique()} stations, {df['parameter'].nunique()} parameters")
df.head(10)

## 2. Data Quality Assessment

Before analysis, check data quality: duplicates, missing values, outliers, temporal gaps.

In [ ]:
from aquascope.analysis.quality import assess_quality, preprocess, print_quality_report

quality_report = assess_quality(df)
print(print_quality_report(quality_report))

In [ ]:
# Apply recommended preprocessing
if quality_report.recommended_steps:
    print(f"Applying: {quality_report.recommended_steps}")
    df_clean = preprocess(df, steps=quality_report.recommended_steps)
    print(f"Before: {len(df):,} rows → After: {len(df_clean):,} rows")
else:
    df_clean = df
    print("Data quality is good — no preprocessing needed.")

## 3. Exploratory Data Analysis (EDA)

Auto-profile the dataset: statistics per parameter, outliers, correlations, completeness.

In [ ]:
from aquascope.analysis.eda import generate_eda_report, print_eda_report, profile_dataset

eda_report = generate_eda_report(df_clean)
print(print_eda_report(eda_report))

In [ ]:
# Show correlation matrix if available
if eda_report.correlation_matrix is not None:
    print("\nCorrelation Matrix:")
    display(eda_report.correlation_matrix)

## 4. AI Methodology Recommendations

Generate a `DatasetProfile` from the EDA and let the AI engine recommend suitable research methodologies.

In [ ]:
from aquascope.ai_engine.recommender import recommend

# Auto-generate profile from data
profile = profile_dataset(df_clean)
profile.research_goal = "Long-term water quality trend analysis and pollution source identification"
profile.keywords = ["trend", "pollution", "multivariate", "monitoring"]

print("Dataset Profile:")
print(f"  Parameters: {profile.parameters}")
print(f"  Records: {profile.n_records:,}")
print(f"  Stations: {profile.n_stations}")
print(f"  Time span: {profile.time_span_years:.1f} years")
print(f"  Scope: {profile.geographic_scope}")

In [ ]:
# Get top 5 recommendations
recs = recommend(profile, top_k=5)

print(f"\n{'='*70}")
print(f"  Top {len(recs)} Recommended Research Methodologies")
print(f"{'='*70}\n")

for i, rec in enumerate(recs, 1):
    m = rec.methodology
    print(f"  {i}. {m.name}  (score: {rec.score})")
    print(f"     Category   : {m.category}")
    print(f"     Complexity : {m.complexity}")
    print(f"     Rationale  : {rec.rationale}")
    print()

## 5. Execute a Methodology Pipeline

Run the top recommended methodology automatically.

In [ ]:
from aquascope.pipelines.model_builder import list_available_pipelines, run_pipeline

print("Available pipelines:", list_available_pipelines())

In [ ]:
# Run correlation analysis
result = run_pipeline("correlation_analysis", df_clean)

print(f"\n{'='*70}")
print(f"  {result.method_name}")
print(f"{'='*70}")
print(f"\n  {result.summary}\n")

if result.metrics.get("top_correlations"):
    print("  Top Correlations:")
    for pair in result.metrics["top_correlations"]:
        print(f"    {pair['param1']} ↔ {pair['param2']}: r = {pair['r']:.3f} (p = {pair['p']:.4f})")

In [ ]:
# Run PCA + Clustering
pca_result = run_pipeline("pca_clustering", df_clean, config={"n_clusters": 3, "n_components": 2})

print(f"\n  {pca_result.method_name}")
print(f"  {pca_result.summary}\n")

if pca_result.metrics:
    print(f"  Explained variance: {pca_result.metrics.get('explained_variance')}")
    print(f"  Cluster sizes: {pca_result.metrics.get('cluster_sizes')}")
    if 'loadings' in pca_result.metrics:
        print("\n  PCA Loadings:")
        for param, loads in pca_result.metrics['loadings'].items():
            print(f"    {param}: {loads}")

In [ ]:
# Run Water Quality Index (RPI)
wqi_result = run_pipeline("wqi_calculation", df_clean)

print(f"\n  {wqi_result.method_name}")
print(f"  {wqi_result.summary}")
print(f"  Category distribution: {wqi_result.metrics.get('category_distribution', {})}")

## 6. Explore the Knowledge Base

Browse all 26 built-in research methodologies.

In [ ]:
from aquascope.ai_engine.knowledge_base import get_all_methodologies, search_methodologies

all_methods = get_all_methodologies()
print(f"Total methodologies: {len(all_methods)}\n")

# Group by category
by_cat = {}
for m in all_methods:
    by_cat.setdefault(m.category, []).append(m)

for cat, methods in sorted(by_cat.items()):
    print(f"[{cat}] ({len(methods)} methods)")
    for m in methods:
        print(f"  • {m.name} ({m.complexity}) — {m.typical_scale}")
    print()

In [ ]:
# Search for specific methodologies
ml_methods = search_methodologies(category="machine_learning")
print(f"Machine Learning methods: {len(ml_methods)}")
for m in ml_methods:
    print(f"  • {m.name}: {m.description[:80]}...")

## Next Steps

- **Try real data**: Replace the synthetic data with `TaiwanMOENVCollector` or `USGSCollector`
- **Use LLM mode**: `recommend_with_llm(profile, model='gpt-4o-mini', api_key='sk-...')`
- **Add your own methods**: See `docs/guides/adding_methodology.md`
- **Contribute data sources**: See `docs/guides/adding_data_source.md`
- **Run from CLI**: `aquascope collect --source usgs --days 30`

For more information, visit: https://github.com/Rekin226/aquascope